# 10 Interval Analysis

Intervals describe race compression: whether cars are bunched in DRS trains, spread into clean-air gaps, or reset by interruptions. This notebook profiles gap dynamics without reading telemetry.

In [1]:
from pathlib import Path
import sys
from datetime import datetime
import json

import pandas as pd
import numpy as np
import plotly.express as px

ROOT = Path.cwd()
while not (ROOT / "configs" / "pipeline_config.yaml").exists() and ROOT.parent != ROOT:
    ROOT = ROOT.parent

SHARED = ROOT / "eda" / "shared" / "scripts"
if str(SHARED) not in sys.path:
    sys.path.insert(0, str(SHARED))

from config import CLEANED_DATA_PATH

NOTEBOOK_NAME = "10_interval_analysis"
OUTPUT_TABLES = ROOT / "eda" / "silver" / "outputs" / "tables" / NOTEBOOK_NAME
OUTPUT_CHARTS = ROOT / "eda" / "silver" / "outputs" / "charts" / NOTEBOOK_NAME
OUTPUT_REPORTS = ROOT / "eda" / "silver" / "outputs" / "reports" / NOTEBOOK_NAME
INSIGHTS = ROOT / "eda" / "silver" / "insights"
CHECKPOINTS = ROOT / "eda" / "silver" / "checkpoints"
for path in [OUTPUT_TABLES, OUTPUT_CHARTS, OUTPUT_REPORTS, INSIGHTS, CHECKPOINTS]:
    path.mkdir(parents=True, exist_ok=True)

def write_report(name: str, payload: dict) -> None:
    (OUTPUT_REPORTS / f"{name}.json").write_text(json.dumps(payload, indent=2, default=str), encoding="utf-8")

def write_insight(title: str, observations: list[str], issues: list[str], recommendations: list[str]) -> None:
    content = f"# {title}\n\n"
    content += f"**Generated at:** {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}\n\n"
    content += "## Key Observations\n\n" + "\n".join(f"- {item}" for item in observations) + "\n\n"
    content += "## Issues\n\n" + ("\n".join(f"- {item}" for item in issues) if issues else "- None") + "\n\n"
    content += "## Recommendations\n\n" + "\n".join(f"- {item}" for item in recommendations) + "\n"
    (INSIGHTS / f"{NOTEBOOK_NAME}.md").write_text(content, encoding="utf-8")

def add_event_type(sessions: pd.DataFrame) -> pd.DataFrame:
    sessions = sessions.copy()
    sessions["event_type"] = np.where(
        sessions["session_name"].astype(str).str.lower().eq("sprint"),
        "SPRINT_RACE",
        "GRAND_PRIX_RACE",
    )
    return sessions

def save_fig(fig, name: str):
    fig.write_html(OUTPUT_CHARTS / f"{name}.html", include_plotlyjs="cdn")
    fig.show()

print("=" * 72)
print(f"SILVER STRATEGY EDA - {NOTEBOOK_NAME}")
print(f"Start time: {datetime.now()}")
print(f"Cleaned data path: {CLEANED_DATA_PATH}")
print("=" * 72)


SILVER STRATEGY EDA - 10_interval_analysis
Start time: 2026-06-02 02:14:19.123869
Cleaned data path: D:\F1_WinRate_Predictor\data\cleaned


In [2]:
intervals = pd.read_parquet(CLEANED_DATA_PATH / "intervals.parquet")
sessions = add_event_type(pd.read_parquet(CLEANED_DATA_PATH / "sessions.parquet"))
drivers = pd.read_parquet(CLEANED_DATA_PATH / "drivers.parquet")
session_result = pd.read_parquet(CLEANED_DATA_PATH / "session_result.parquet")
intervals["date"] = pd.to_datetime(intervals["date"], errors="coerce", utc=True)
for column in ["interval", "gap_to_leader", "driver_number", "session_key"]:
    if column in intervals.columns:
        intervals[column] = pd.to_numeric(intervals[column], errors="coerce")
intervals = intervals.merge(sessions[["session_key", "year", "event_type", "circuit_short_name", "date_start", "date_end"]], on="session_key", how="left")
intervals["date_start"] = pd.to_datetime(intervals["date_start"], errors="coerce", utc=True)
intervals["date_end"] = pd.to_datetime(intervals["date_end"], errors="coerce", utc=True)
duration_seconds = (intervals["date_end"] - intervals["date_start"]).dt.total_seconds()
intervals["race_phase_pct"] = ((intervals["date"] - intervals["date_start"]).dt.total_seconds() / duration_seconds * 100).clip(0, 100)

coverage = pd.DataFrame([{
    "rows": len(intervals),
    "sessions": intervals["session_key"].nunique(),
    "circuits": intervals["circuit_short_name"].nunique(),
    "valid_interval_rows": int(intervals["interval"].notna().sum()),
}])
coverage.to_csv(OUTPUT_TABLES / "interval_coverage.csv", index=False)
display(coverage)

,rows,sessions,circuits,valid_interval_rows
0,1392480,68,24,1375929


## 1. Gap Distribution

The interval distribution separates DRS-range battles from clean-air separation. Values between roughly 0.5s and 1.5s are treated as potential DRS-train conditions, not exact FIA DRS eligibility.

In [3]:
valid_intervals = intervals[intervals["interval"].between(0, 30)].copy()
interval_stats = valid_intervals["interval"].describe(percentiles=[0.1, 0.25, 0.5, 0.75, 0.9, 0.95]).reset_index()
interval_stats.columns = ["metric", "interval_seconds"]
interval_stats.to_csv(OUTPUT_TABLES / "interval_distribution_stats.csv", index=False)
display(interval_stats)

fig = px.histogram(
    valid_intervals.sample(min(200_000, len(valid_intervals)), random_state=42),
    x="interval",
    color="event_type",
    nbins=80,
    title="Car-to-Car Interval Distribution",
)
save_fig(fig, "interval_distribution")

,metric,interval_seconds
0,count,1.354519e+06
1,mean,3.761503e+00
2,std,4.726529e+00
3,min,0.000000e+00
4,10%,5.540000e-01
5,25%,8.880000e-01
6,50%,1.945000e+00
7,75%,4.495500e+00
8,90%,9.442000e+00
9,95%,1.410400e+01


In [4]:
phase_gap = valid_intervals.groupby(["session_key", "event_type", "circuit_short_name"], as_index=False).agg(
    median_interval=("interval", "median"),
    p25_interval=("interval", lambda s: s.quantile(0.25)),
    drs_train_rate=("interval", lambda s: s.between(0.5, 1.5).mean()),
    compressed_rate=("interval", lambda s: s.le(1.5).mean()),
    observations=("interval", "count"),
)
phase_gap.to_csv(OUTPUT_TABLES / "session_gap_compression.csv", index=False)
display(phase_gap.sort_values("drs_train_rate", ascending=False).head(20))

fig = px.scatter(
    phase_gap,
    x="median_interval",
    y="drs_train_rate",
    size="observations",
    color="event_type",
    hover_name="circuit_short_name",
    title="Race Compression: Median Interval vs DRS-Train Rate",
)
save_fig(fig, "race_compression_scatter")

,session_key,event_type,circuit_short_name,median_interval,p25_interval,drs_train_rate,compressed_rate,observations
45,9934,SPRINT_RACE,Spa-Francorchamps,0.8310,0.6080,0.632065,0.765336,7466
38,9883,SPRINT_RACE,Austin,1.1590,0.7850,0.582614,0.640567,6488
4,9506,SPRINT_RACE,Miami,0.8360,0.5860,0.559655,0.713072,7887
25,9654,SPRINT_RACE,Lusail,1.1330,0.7470,0.558556,0.647169,8035
19,9616,SPRINT_RACE,Austin,1.1015,0.6920,0.545057,0.660615,7546
10,9549,SPRINT_RACE,Spielberg,0.8800,0.6430,0.542259,0.667823,8649
32,9845,SPRINT_RACE,Lusail,1.1890,0.7460,0.526618,0.604189,8547
35,9864,SPRINT_RACE,Interlagos,0.9950,0.5980,0.522638,0.682475,8371
28,9672,SPRINT_RACE,Shanghai,0.9480,0.5860,0.504959,0.671261,7763
61,11240,SPRINT_RACE,Shanghai,0.8810,0.5310,0.479513,0.706263,8957


## 2. When Gaps Form

Since intervals are timestamped rather than lap-numbered, we use normalized race phase to estimate whether fields spread early or remain compressed deep into the session.

In [5]:
valid_intervals = valid_intervals.copy()
valid_intervals["phase_bucket"] = pd.cut(valid_intervals["race_phase_pct"], bins=[-1, 20, 40, 60, 80, 101], labels=["0-20%", "20-40%", "40-60%", "60-80%", "80-100%"])
phase_compression = valid_intervals.groupby(["event_type", "phase_bucket"], observed=True, as_index=False).agg(
    median_interval=("interval", "median"),
    drs_train_rate=("interval", lambda s: s.between(0.5, 1.5).mean()),
    observations=("interval", "count"),
)
phase_compression.to_csv(OUTPUT_TABLES / "interval_phase_compression.csv", index=False)
display(phase_compression)

fig = px.line(
    phase_compression,
    x="phase_bucket",
    y="median_interval",
    color="event_type",
    markers=True,
    title="Gap Formation Across Race Phase",
)
save_fig(fig, "gap_formation_by_phase")

,event_type,phase_bucket,median_interval,drs_train_rate,observations
0,GRAND_PRIX_RACE,0-20%,1.003,0.526974,252635
1,GRAND_PRIX_RACE,20-40%,2.095,0.301731,319208
2,GRAND_PRIX_RACE,40-60%,2.884,0.241176,313066
3,GRAND_PRIX_RACE,60-80%,3.175,0.235633,268468
4,GRAND_PRIX_RACE,80-100%,2.540,0.279114,79953
5,SPRINT_RACE,0-20%,0.749,0.590276,26120
6,SPRINT_RACE,20-40%,1.111,0.531854,40842
7,SPRINT_RACE,40-60%,1.474,0.418330,31915
8,SPRINT_RACE,60-80%,0.900,0.599348,6749
9,SPRINT_RACE,80-100%,1.314,0.483840,15563


## 3. Race Classification

Sessions are classified as compressed, balanced, or processional using median interval and DRS-train rate. This is an analytical label for feature design, not an official race classification.

In [6]:
def classify_race(row):
    if row["drs_train_rate"] >= 0.20:
        return "DRS_TRAIN_HEAVY"
    if row["median_interval"] >= 3.0:
        return "PROCESSIONAL_SPREAD"
    return "BALANCED_RACE"

race_class = phase_gap.copy()
race_class["race_gap_class"] = race_class.apply(classify_race, axis=1)
race_class.to_csv(OUTPUT_TABLES / "race_gap_classification.csv", index=False)
display(race_class["race_gap_class"].value_counts().reset_index().rename(columns={"index": "race_gap_class", "race_gap_class": "sessions"}))

fig = px.bar(
    race_class["race_gap_class"].value_counts().reset_index(),
    x="race_gap_class",
    y="count",
    title="Race Gap Classification",
)
save_fig(fig, "race_gap_classification")

,sessions,count
0,DRS_TRAIN_HEAVY,67
1,PROCESSIONAL_SPREAD,1


In [7]:
circuit_gap = race_class.groupby("circuit_short_name", as_index=False).agg(
    sessions=("session_key", "nunique"),
    avg_median_interval=("median_interval", "mean"),
    avg_drs_train_rate=("drs_train_rate", "mean"),
    avg_compressed_rate=("compressed_rate", "mean"),
).sort_values("avg_drs_train_rate", ascending=False)
circuit_gap.to_csv(OUTPUT_TABLES / "circuit_gap_profiles.csv", index=False)
display(circuit_gap)

fig = px.bar(
    circuit_gap,
    x="avg_drs_train_rate",
    y="circuit_short_name",
    orientation="h",
    color="avg_median_interval",
    title="Circuit DRS-Train Profile",
)
save_fig(fig, "circuit_drs_train_profile")

,circuit_short_name,sessions,avg_median_interval,avg_drs_train_rate,avg_compressed_rate
8,Lusail,4,1.387250,0.477842,0.550026
19,Spa-Francorchamps,3,1.507167,0.446729,0.543938
5,Interlagos,4,1.463500,0.422442,0.532677
0,Austin,4,1.894875,0.410544,0.481819
11,Miami,6,1.654000,0.405611,0.497072
12,Monte Carlo,2,1.532500,0.392825,0.493392
16,Shanghai,6,1.515000,0.392296,0.527272
23,Zandvoort,2,1.952000,0.358678,0.455108
21,Suzuka,3,1.933667,0.345978,0.417594
20,Spielberg,3,2.063000,0.344541,0.434607


## Final Interval Report

In [8]:
top_train = circuit_gap.iloc[0].to_dict() if len(circuit_gap) else {}
report = {
    "notebook": NOTEBOOK_NAME,
    "timestamp": datetime.now().isoformat(),
    "status": "PASS",
    "interval_rows": int(len(intervals)),
    "valid_interval_rows": int(len(valid_intervals)),
    "sessions": int(intervals["session_key"].nunique()),
    "avg_drs_train_rate": float(race_class["drs_train_rate"].mean()) if len(race_class) else 0.0,
    "highest_drs_train_circuit": top_train.get("circuit_short_name"),
    "highest_drs_train_rate": float(top_train.get("avg_drs_train_rate", 0)),
}
write_report("interval_analysis", report)
write_insight(
    "Silver Interval Analysis Insights",
    [
        f"Analyzed {report['valid_interval_rows']:,} valid interval observations.",
        f"Average session DRS-train rate: {report['avg_drs_train_rate']:.3f}.",
        f"Highest DRS-train circuit: {report['highest_drs_train_circuit']}.",
    ],
    [],
    [
        "Use session-level gap class as a Gold race-context feature.",
        "Combine DRS-train rate with overtaking volume to distinguish passable tracks from traffic traps.",
        "Use normalized race phase for interval features because raw interval data is timestamped rather than lap-indexed.",
    ],
)
(CHECKPOINTS / "silver_interval_analysis_completed.txt").write_text(json.dumps(report, indent=2), encoding="utf-8")
print(report)

{'notebook': '10_interval_analysis', 'timestamp': '2026-06-02T02:14:21.974391', 'status': 'PASS', 'interval_rows': 1392480, 'valid_interval_rows': 1354519, 'sessions': 68, 'avg_drs_train_rate': 0.3585296854904298, 'highest_drs_train_circuit': 'Lusail', 'highest_drs_train_rate': 0.47784202203855974}
